In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "bioGraph").is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pickle
from tqdm import tqdm
from scipy import sparse

from bioGraph.data.loading import load_disease_genes, load_ppi_graph
from bioGraph.data.splitting import split_disease_genes
from bioGraph.methods.ranking import (
    qa_score,
    
    dk_score,
    rwr_score,
    diamond_score,
    neighbourhood_score,
    normalize_adjacency,
    )
from bioGraph.evaluation.metrics import average_precision_at_k, recall_at_k
from bioGraph.methods.utils import scores_to_ranking
from bioGraph.sim import run_benchmark_simulation, validate_benchmark_results

In [ ]:
data_dir = project_root / "data" / "raw"
results_dir = project_root / "outputs" / "results"
reports_dir = project_root / "outputs" / "reports"
results_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
ppi_path = data_dir / "PPI202207.txt"
disease_path = data_dir / "pcbi.1004120.s004.txt"

Ggen = load_ppi_graph(ppi_path)
diseases = load_disease_genes(disease_path)
nodelist = list(Ggen.nodes())


print(
    f"Loaded Ggen with {Ggen.number_of_nodes():,} nodes and "
    f"{Ggen.number_of_edges():,} undirected edges."
)
print(f"Average coordination number: {2 * Ggen.number_of_edges() / Ggen.number_of_nodes():.2f}")

print(f"Loaded diseases for {len(diseases)} diseases.")
print(f"Average number of genes per disease: {np.mean([len(genes) for genes in diseases.values()]):.2f}")
print(f"Example: alzheimer disease has {len(diseases['breast neoplasms'])} genes.")


In [ ]:
# Example:
disease_name = "breast neoplasms"#"nutritional and metabolic diseases"#"cardiomyopathies"#"breast neoplasms"
train_genes, test_genes = split_disease_genes(disease_name, 0.5,diseases)
print(f"Train: {len(train_genes)} genes")
print(f"Test: {len(test_genes)} genes")

In [ ]:
# Pre-compute basic matrices with explicit graph node order.
# Using range(n) would be incorrect here because Entrez node IDs are not 0..n-1.
n = len(nodelist)

L = nx.laplacian_matrix(Ggen, nodelist=nodelist)
A_sparse = nx.adjacency_matrix(Ggen, nodelist=nodelist)
A_sparse = sparse.csr_matrix(A_sparse)
R = normalize_adjacency(Ggen, A_sparse, nodelist=nodelist)

# Hyperparameters (updated)
qa_t = 0.45
qa_diag = 5
dk_t = 0.3
rwr_return_prob = 0.4
diamond_alpha = 9
diamond_number_to_rank = 300

#running times
# RWR ~ NBR  << DK  < DIAMOND ~ QA

# Run each method directly from bioGraph.methods.ranking.
qa_scores = qa_score(Ggen, train_genes, t=qa_t, H=A_sparse.copy(), diag=qa_diag, nodelist=nodelist)
dk_scores = dk_score(Ggen, train_genes, t=dk_t, L=L, nodelist=nodelist)
rwr_scores = rwr_score(
    Ggen,
    train_genes,
    normalized_adjacency=R,
    return_prob=rwr_return_prob,
    nodelist=nodelist,
)
diamond_scores = diamond_score(
    Ggen,
    train_genes,
    A=A_sparse,
    alpha=diamond_alpha,
    number_to_rank=diamond_number_to_rank,
    nodelist=nodelist,
)
neighbourhood_scores = neighbourhood_score(
    Ggen,
    train_genes,
    A=A_sparse,
    nodelist=nodelist,
)

rankings = {
    "qa_score": scores_to_ranking(qa_scores, nodelist, Ggen, train_genes),
    "dk_score": scores_to_ranking(dk_scores, nodelist, Ggen, train_genes),
    "rwr_score": scores_to_ranking(rwr_scores, nodelist, Ggen, train_genes),
    "diamond_score": scores_to_ranking(diamond_scores, nodelist, Ggen, train_genes),
    "neighbourhood_score": scores_to_ranking(neighbourhood_scores, nodelist, Ggen, train_genes),
}

k_values = [5, 10, 20]

print(f"n={n} | Train genes: {len(train_genes)} | Test genes: {len(test_genes)}")
print("\n=== Benchmark results on current test set ===")
for method_name, ranking in rankings.items():
    print(f"\nMethod: {method_name}")
    for k in k_values:
        ap = average_precision_at_k(ranking, test_genes, k=k)
        rec = recall_at_k(ranking, test_genes, k=k)
        print(f"K={k:>3} | AP@K={ap:.4f} | Recall@K={rec:.4f}")

    ap_all = average_precision_at_k(ranking, test_genes, k=len(ranking))
    rec_all = recall_at_k(ranking, test_genes, k=len(ranking))
    print(f"K=ALL | AP@All={ap_all:.4f} | Recall@All={rec_all:.4f}")

    top5 = ranking[:5]
    top5_hits = [row for row in top5 if row["gene_id"] in set(test_genes)]
    print("Top 5 predictions:")
    for row in top5:
        print(row)
    print(f"Top-5 test hits: {len(top5_hits)}")

In [ ]:
# aNBR (absolute), rNBR (relative), and RWR benchmark
# Setup: disease='breast neoplasms', N=30 runs, constant split fraction.

disease_name = "breast neoplasms"
split_fraction = 0.5
num_runs = 30
base_seed = 0
k_values_eval = [20, 100]

results_runs = []

for run_idx in range(num_runs):
    split_seed = base_seed + run_idx
    train_genes_run, test_genes_run = split_disease_genes(
        disease_name,
        split_fraction,
        diseases_dict=diseases,
        random_state=split_seed,
    )

    # aNBR: absolute number of seed-neighbour connections
    anbr_scores_run = neighbourhood_score(
        Ggen,
        train_genes_run,
        A=A_sparse,
        nodelist=nodelist,
        weighted="absolute",
    )
    anbr_ranking_run = scores_to_ranking(anbr_scores_run, nodelist, Ggen, train_genes_run)

    # rNBR: normalized by node degree
    rnbr_scores_run = neighbourhood_score(
        Ggen,
        train_genes_run,
        A=A_sparse,
        nodelist=nodelist,
        weighted="relative",
    )
    rnbr_ranking_run = scores_to_ranking(rnbr_scores_run, nodelist, Ggen, train_genes_run)

    # RWR reference
    rwr_scores_run = rwr_score(
        Ggen,
        train_genes_run,
        normalized_adjacency=R,
        return_prob=rwr_return_prob,
        nodelist=nodelist,
    )
    rwr_ranking_run = scores_to_ranking(rwr_scores_run, nodelist, Ggen, train_genes_run)

    row = {"seed": split_seed}
    for k_eval in k_values_eval:
        row[f"aNBR_AP@{k_eval}"] = average_precision_at_k(anbr_ranking_run, test_genes_run, k=k_eval)
        row[f"aNBR_Recall@{k_eval}"] = recall_at_k(anbr_ranking_run, test_genes_run, k=k_eval)
        row[f"rNBR_AP@{k_eval}"] = average_precision_at_k(rnbr_ranking_run, test_genes_run, k=k_eval)
        row[f"rNBR_Recall@{k_eval}"] = recall_at_k(rnbr_ranking_run, test_genes_run, k=k_eval)
        row[f"RWR_AP@{k_eval}"] = average_precision_at_k(rwr_ranking_run, test_genes_run, k=k_eval)
        row[f"RWR_Recall@{k_eval}"] = recall_at_k(rwr_ranking_run, test_genes_run, k=k_eval)

    results_runs.append(row)


def _method_stats(results, method, metric, k_eval):
    vals = np.array([row[f"{method}_{metric}@{k_eval}"] for row in results], dtype=float)
    return vals.mean(), vals.std()


def print_k_table(results, k_eval):
    print(f"\nResults for K={k_eval}")
    print(
        "seed | "
        " aNBR AP  aNBR Rec | "
        " rNBR AP  rNBR Rec | "
        "  RWR AP   RWR Rec"
    )
    print("-----+-------------------+-------------------+-----------------")

    for row in results:
        print(
            f"{row['seed']:>4} | "
            f"{row[f'aNBR_AP@{k_eval}']:.4f}   {row[f'aNBR_Recall@{k_eval}']:.4f}  | "
            f"{row[f'rNBR_AP@{k_eval}']:.4f}   {row[f'rNBR_Recall@{k_eval}']:.4f}  | "
            f"{row[f'RWR_AP@{k_eval}']:.4f}   {row[f'RWR_Recall@{k_eval}']:.4f}"
        )


def print_summary_table(results, k_values):
    print("\nSummary (mean +/- std)")
    print("K   | method |       AP (mean +/- std) |   Recall (mean +/- std)")
    print("----+--------+--------------------------+------------------------")

    for k_eval in k_values:
        for method in ["aNBR", "rNBR", "RWR"]:
            ap_mean, ap_std = _method_stats(results, method, "AP", k_eval)
            rec_mean, rec_std = _method_stats(results, method, "Recall", k_eval)
            print(
                f"{k_eval:>3} | {method:<6} | "
                f"{ap_mean:>7.4f} +/- {ap_std:<7.4f} | "
                f"{rec_mean:>7.4f} +/- {rec_std:<7.4f}"
            )


# for k_eval in k_values_eval:
#     print_k_table(results_runs, k_eval)

print_summary_table(results_runs, k_values_eval)

## Data generation

In [ ]:
# Generate the benchmark artifact through the tested package function.
disease_set = [
    "breast neoplasms",
    "cardiomyopathies",
    "nutritional and metabolic diseases",
    "adrenal gland diseases",
    "arrhythmias cardiac",
    "cerebrovascular disorders",
]
method_set = ["aNBR", "rNBR", "RWR", "DK", "QA+", "QA-", "DIAMOND"]
pickle_path = results_dir / "results_methods_by_6disease.pkl"

benchmark_results = run_benchmark_simulation(
    Ggen,
    diseases,
    pickle_path,
    disease_set=disease_set,
    method_set=method_set,
    num_runs=30,
    split_fraction=0.5,
    base_seed=0,
    rwr_return_prob=0.4,
    qa_t=0.45,
    qa_diag=5,
    dk_t=0.3,
    diamond_alpha=9,
    diamond_number_to_rank=300,
)

print(f"Saved benchmark results to: {pickle_path}")
print(f"Total entries: {len(benchmark_results['runs'])}")

## Data analysis

In [ ]:
# Load generated benchmark artifact and summarize AP/Recall at K=25 and K=300.
# `pickle_path` is set by the generation cell above.
with pickle_path.open("rb") as handle:
    benchmark_results_loaded = pickle.load(handle)
validate_benchmark_results(benchmark_results_loaded)

method_set = benchmark_results_loaded["config"]["method_set"]
disease_set = benchmark_results_loaded["config"]["disease_set"]
runs = benchmark_results_loaded["runs"]
k_values_interest = [25, 300]


def format_mean_std_to_first_sig_digit(mean_value, std_value):
    """
    Round std to one significant digit and format mean with matching precision.
    Returns a string like '0.010 +/- 0.009'.
    """
    std_value = float(std_value)
    mean_value = float(mean_value)

    if std_value == 0.0:
        # Keep a stable default precision when variability is zero.
        return f"{mean_value:.4f} +/- {0.0:.4f}"

    order = int(np.floor(np.log10(abs(std_value))))
    decimals = max(0, -order)

    std_rounded = round(std_value, decimals)
    mean_rounded = round(mean_value, decimals)

    mean_str = f"{mean_rounded:.{decimals}f}"
    std_str = f"{std_rounded:.{decimals}f}"
    return f"{mean_str} +/- {std_str}"


def summarize_metric_table(metric_name, k_eval):
    """Return table[ disease ][ method ] = formatted mean +/- std string."""
    table = {disease_name: {} for disease_name in disease_set}

    for disease_name in disease_set:
        disease_runs = [row for row in runs if row["disease"] == disease_name]

        for method_name in method_set:
            metric_values = []

            for row in disease_runs:
                scores = np.asarray(row["scores"][method_name], dtype=float)
                ranking = scores_to_ranking(scores, nodelist, Ggen, row["train_genes"])

                if metric_name == "AP":
                    metric_value = average_precision_at_k(ranking, row["test_genes"], k=k_eval)
                elif metric_name == "Recall":
                    metric_value = recall_at_k(ranking, row["test_genes"], k=k_eval)
                else:
                    raise ValueError(f"Unknown metric_name: {metric_name}")

                metric_values.append(metric_value)

            values_arr = np.asarray(metric_values, dtype=float)
            mean_value = values_arr.mean()
            std_value = values_arr.std(ddof=0)
            table[disease_name][method_name] = format_mean_std_to_first_sig_digit(mean_value, std_value)

    return table


def render_table(title, table, methods):
    """Return a pretty disease x method table as a string."""
    lines = []
    lines.append("")
    lines.append(title)
    disease_col_width = max(len("Disease"), max(len(d) for d in table))
    method_col_width = {m: max(len(m), 16) for m in methods}

    header = "Disease".ljust(disease_col_width) + " | " + " | ".join(
        m.ljust(method_col_width[m]) for m in methods
    )
    sep = "-" * len(header)
    lines.append(header)
    lines.append(sep)

    for disease_name in table:
        row_values = [table[disease_name][m].ljust(method_col_width[m]) for m in methods]
        lines.append(disease_name.ljust(disease_col_width) + " | " + " | ".join(row_values))

    return "\n".join(lines)


table_ap_25 = summarize_metric_table("AP", 25)
table_ap_300 = summarize_metric_table("AP", 300)
table_recall_25 = summarize_metric_table("Recall", 25)
table_recall_300 = summarize_metric_table("Recall", 300)

text_blocks = [
    render_table("AP@25 (mean +/- std)", table_ap_25, method_set),
    render_table("AP@300 (mean +/- std)", table_ap_300, method_set),
    render_table("Recall@25 (mean +/- std)", table_recall_25, method_set),
    render_table("Recall@300 (mean +/- std)", table_recall_300, method_set),
]
full_report = "\n\n".join(text_blocks)

print(full_report)

report_path = reports_dir / "results_methods_summary_tables.txt"
report_path.write_text(full_report + "\n", encoding="utf-8")
print(f"\nSaved summary tables to: {report_path}")

analysis_results = {
    "AP@25": table_ap_25,
    "AP@300": table_ap_300,
    "Recall@25": table_recall_25,
    "Recall@300": table_recall_300,
    "report_path": str(report_path),
}

In [ ]:
# Top-25 detection probabilities and recurrent false positives.
# Counts use the saved runs and scores; no prioritization method is rerun.
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display

top_k_hits = 25
minimum_false_positive_occurrences = 29


def _gene_symbol(gene_id):
    return Ggen.nodes[gene_id].get("symbol", "") if gene_id in Ggen else ""


def _tp_probability_table(hit_counts, test_opportunities):
    rows = []
    for gene_id, opportunity_count in test_opportunities.items():
        row = {
            "gene_id": int(gene_id),
            "symbol": _gene_symbol(gene_id),
            #"testing_set_runs": int(opportunity_count),
        }
        probabilities = []
        for method_name in method_set:
            detections = int(hit_counts[gene_id][method_name])
            probability = detections / opportunity_count
            #row[f"{method_name}_detections"] = detections
            row[f"{method_name}_probability"] = probability
            probabilities.append(probability)
        row["max_detection_probability"] = max(probabilities, default=0.0)
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["max_detection_probability", "gene_id"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )


def _recurrent_false_positive_table(false_positive_counts, minimum_occurrences):
    rows = []
    for gene_id, counts_by_method in false_positive_counts.items():
        total_occurrences = int(sum(counts_by_method.values()))
        if total_occurrences < minimum_occurrences:
            continue
        methods = [method for method in method_set if counts_by_method[method] > 0]
        rows.append(
            {
                "gene_id": int(gene_id),
                "symbol": _gene_symbol(gene_id),
                "false_positive_occurrences": total_occurrences,
                "algorithm_count": len(methods),
                "algorithms": ", ".join(methods),
                "occurrences_by_algorithm": ", ".join(
                    f"{method}: {counts_by_method[method]}" for method in methods
                ),
            }
        )

    columns = [
        "gene_id",
        "symbol",
        "false_positive_occurrences",
        "algorithm_count",
        "algorithms",
        "occurrences_by_algorithm",
    ]
    return (
        pd.DataFrame(rows, columns=columns)
        .sort_values(
            ["false_positive_occurrences", "algorithm_count", "gene_id"],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )


top25_tp_probability_tables = {}
top25_recurrent_false_positive_tables = {}

for disease_name in disease_set:
    true_positive_counts = defaultdict(Counter)
    false_positive_counts = defaultdict(Counter)
    test_opportunities = Counter()
    disease_runs = [row for row in runs if row["disease"] == disease_name]

    for row in disease_runs:
        testing_genes = set(row["test_genes"])
        test_opportunities.update(testing_genes)

        for method_name in method_set:
            ranking = scores_to_ranking(
                row["scores"][method_name],
                nodelist,
                Ggen,
                row["train_genes"],
            )
            for prediction in ranking[:top_k_hits]:
                gene_id = prediction["gene_id"]
                if gene_id in testing_genes:
                    true_positive_counts[gene_id][method_name] += 1
                else:
                    false_positive_counts[gene_id][method_name] += 1

    # Table 1: P(top-25 detection | gene was in this run's testing set).
    top25_tp_probability_tables[disease_name] = _tp_probability_table(
        true_positive_counts, test_opportunities
    )

    # Table 2: non-testing genes repeatedly predicted in the top 25.
    top25_recurrent_false_positive_tables[disease_name] = (
        _recurrent_false_positive_table(
            false_positive_counts, minimum_false_positive_occurrences
        )
    )

    print(f"\n{'=' * 100}\nDisease: {disease_name}")
    print(
        f"Table 1: top-{top_k_hits} detection probability conditional on "
        "membership in the testing set"
    )
    display(top25_tp_probability_tables[disease_name])
    print(
        f"Table 2: false-positive genes appearing in the top {top_k_hits} "
        f"at least {minimum_false_positive_occurrences} times"
    )
    display(top25_recurrent_false_positive_tables[disease_name])


In [ ]:
# Build MAP@K and Recall@K curves from the loaded benchmark runs.
k_values_curve = np.unique(np.geomspace(1, len(nodelist), 20).astype(int))
results = {"k_values": k_values_curve, "by_method": {}}

for method_name in method_set:
    ap_runs = np.zeros((len(runs), len(k_values_curve)), dtype=float)
    recall_runs = np.zeros_like(ap_runs)
    for run_index, row in enumerate(runs):
        ranking = scores_to_ranking(
            row["scores"][method_name], nodelist, Ggen, row["train_genes"]
        )
        for k_index, k_eval in enumerate(k_values_curve):
            ap_runs[run_index, k_index] = average_precision_at_k(
                ranking, row["test_genes"], k=int(k_eval)
            )
            recall_runs[run_index, k_index] = recall_at_k(
                ranking, row["test_genes"], k=int(k_eval)
            )
    results["by_method"][method_name] = {
        "ap_mean": ap_runs.mean(axis=0),
        "ap_std": ap_runs.std(axis=0),
        "recall_mean": recall_runs.mean(axis=0),
        "recall_std": recall_runs.std(axis=0),
    }

# Plot MAP@K and Recall@K with +/- 1 std tubes for all methods.
k_vals = results["k_values"]
by_method = results["by_method"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
ax_ap, ax_recall = axes

for method_name, values in by_method.items():
    map_mean = values["ap_mean"]
    map_std = values["ap_std"]
    recall_mean = values["recall_mean"]
    recall_std = values["recall_std"]

    ax_ap.plot(k_vals, map_mean, linewidth=2, label=f"{method_name}")
    ax_ap.fill_between(
        k_vals,
        np.clip(map_mean - map_std, 0.0, 1.0),
        np.clip(map_mean + map_std, 0.0, 1.0),
        alpha=0.2,
    )

    ax_recall.plot(k_vals, recall_mean, linewidth=2, label=f"{method_name}")
    ax_recall.fill_between(
        k_vals,
        np.clip(recall_mean - recall_std, 0.0, 1.0),
        np.clip(recall_mean + recall_std, 0.0, 1.0),
        alpha=0.2,
    )

ax_ap.set_title("MAP@K")
ax_recall.set_title("Recall@K")

for ax in axes:
    ax.set_xlabel("K")
    ax.set_xlim(1, int(k_vals[-1]))
    ax.set_xscale("log")
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", title="Method")

ax_ap.set_ylabel("Metric value")

plt.tight_layout()
plt.show()

In [ ]:
def evaluate_diffusion_over_time(
    disease_name,
    split_fraction,
    num_runs,
    time_values,
    k=20,
    diseases_dict=None,
    graph=None,
    random_seed=None,
):
    """
    Evaluate diffusion ranking over a grid of diffusion times.

    For each run, a new random train/test split is created. For each diffusion
    time t, the diffusion-based ranking is computed from the training genes,
    then AP@k and Recall@k are evaluated against the test genes.

    Parameters
    ----------
    disease_name : str
        Disease key in diseases_dict.
    split_fraction : float
        Fraction of genes used for training in each run.
    num_runs : int
        Number of random seeds / random splits.
    time_values : iterable of float
        Diffusion times to evaluate.
    k : int, optional
        Cutoff for AP@k and Recall@k.
    diseases_dict : dict, optional
        Disease-to-genes mapping. If None, uses global diseases.
    graph : networkx.Graph, optional
        Graph to score on. If None, uses global Ggen.
    random_seed : int, optional
        Base seed for reproducible repeated runs.

    Returns
    -------
    results : dict
        Contains time_values, ap_runs, recall_runs, ap_mean, ap_std,
        recall_mean, recall_std.
    """
    if diseases_dict is None:
        diseases_dict = diseases
    if graph is None:
        graph = Ggen

    time_values = np.asarray(list(time_values), dtype=float)
    if time_values.ndim != 1 or time_values.size == 0:
        raise ValueError("time_values must be a non-empty 1D iterable.")
    if num_runs <= 0:
        raise ValueError("num_runs must be a positive integer.")
    if k <= 0:
        raise ValueError("k must be a positive integer.")
    if disease_name not in diseases_dict:
        raise ValueError(f"Disease '{disease_name}' not found.")

    ap_runs = np.zeros((num_runs, len(time_values)), dtype=float)
    recall_runs = np.zeros((num_runs, len(time_values)), dtype=float)
    nodelist_local = list(graph.nodes())
    laplacian = nx.laplacian_matrix(graph, nodelist=nodelist_local)

    for run_idx in range(num_runs):
        run_seed = None if random_seed is None else random_seed + run_idx
        train_genes, test_genes = split_disease_genes(
            disease_name,
            split_fraction,
            diseases_dict=diseases_dict,
            random_state=run_seed,
        )

        for time_idx, t_value in enumerate(time_values):
            scores = dk_score(
                graph,
                train_genes,
                t=float(t_value),
                L=laplacian,
                nodelist=nodelist_local,
            )
            ranking = scores_to_ranking(
                scores, nodelist_local, graph, train_genes
            )
            ap_runs[run_idx, time_idx] = average_precision_at_k(ranking, test_genes, k=k)
            recall_runs[run_idx, time_idx] = recall_at_k(ranking, test_genes, k=k)

    results = {
        "time_values": time_values,
        "k": k,
        "ap_runs": ap_runs,
        "recall_runs": recall_runs,
        "ap_mean": ap_runs.mean(axis=0),
        "ap_std": ap_runs.std(axis=0),
        "recall_mean": recall_runs.mean(axis=0),
        "recall_std": recall_runs.std(axis=0),
    }
    return results


# Example: diffusion time sweep from 0 to 3 with 20 steps
# Replace num_runs if you want more or fewer random seeds.
diffusion_time_results = evaluate_diffusion_over_time(
    disease_name="breast neoplasms",
    split_fraction=0.5,
    num_runs=50,
    time_values=np.linspace(0.0, 3.0, 20),
    k=20,
    random_seed=42,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
ax_map, ax_recall = axes

t_vals = diffusion_time_results["time_values"]
map_mean = diffusion_time_results["ap_mean"]
map_std = diffusion_time_results["ap_std"]
recall_mean = diffusion_time_results["recall_mean"]
recall_std = diffusion_time_results["recall_std"]

ax_map.plot(t_vals, map_mean, color="tab:blue", linewidth=2, label="Mean MAP@20")
ax_map.fill_between(
    t_vals,
    np.clip(map_mean - map_std, 0.0, 1.0),
    np.clip(map_mean + map_std, 0.0, 1.0),
    color="tab:blue",
    alpha=0.2,
    label="± 1 std",
)
ax_map.set_title("Diffusion MAP@20 vs time")
ax_map.set_xlabel("Diffusion time t")
ax_map.set_ylabel("MAP@20")
ax_map.set_ylim(0, 1.0)
ax_map.grid(True, alpha=0.3)
ax_map.legend(loc="best")

ax_recall.plot(t_vals, recall_mean, color="tab:orange", linewidth=2, label="Mean Recall@20")
ax_recall.fill_between(
    t_vals,
    np.clip(recall_mean - recall_std, 0.0, 1.0),
    np.clip(recall_mean + recall_std, 0.0, 1.0),
    color="tab:orange",
    alpha=0.2,
    label="± 1 std",
)
ax_recall.set_title("Diffusion Recall@20 vs time")
ax_recall.set_xlabel("Diffusion time t")
ax_recall.set_ylabel("Recall@20")
ax_recall.set_ylim(0, 1.0)
ax_recall.grid(True, alpha=0.3)
ax_recall.legend(loc="best")

plt.tight_layout()
plt.show()

In [ ]:
# Legacy duplicate benchmark removed; use the package simulation above.

In [ ]:
# Legacy dense reference implementations were replaced by tests/equivalence.